In [1]:
import os
import zipfile
import pyarrow.parquet as pq
import pandas as pd

folder = r"F:\sanazi\python\data science\code\tamrin\proje\Tennis Schema\Tennis Schema\tennis_data"

data = {}

for file in os.listdir(folder): 
    if file.endswith(".zip"): 
        zip_path = os.path.join(folder, file) 

        with zipfile.ZipFile(zip_path) as z: 
            for parquet_file in z.namelist():
                if parquet_file.endswith(".parquet"): 

                    with z.open(parquet_file) as f: 
                        df = pq.read_table(f).to_pandas()
                    table_name =os.path.basename(os.path.dirname(parquet_file))[4:-8] + os.path.basename(parquet_file)[:-17] 

                    if table_name not in data:
                        data[table_name] = [] 
                    data[table_name].append(df) 


for table_name in data:
    data[table_name] = pd.concat(data[table_name], ignore_index=True)

print(data.keys())

C:\Users\sepehr\AppData\Local\Temp\ipykernel_8684\661795369.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data[table_name] = pd.concat(data[table_name], ignore_index=True)


dict_keys(['matchaway_team', 'matchaway_team_score', 'matchevent', 'matchhome_team', 'matchhome_team_score', 'matchround', 'matchseason', 'matchtime', 'matchtournament', 'matchvenue', 'oddsodds', 'point_by_pointpbp', 'statisticsstatistics', 'tennis_powerpower', 'votesvotes'])


In [2]:
event = data["matchevent"]

print(event.columns)

Index(['match_id', 'first_to_serve', 'home_team_seed', 'away_team_seed',
       'custom_id', 'winner_code', 'default_period_count', 'start_datetime',
       'match_slug', 'final_result_only'],
      dtype='object')


In [3]:
rounds = data["matchround"]

print(rounds.columns)
print(rounds["name"].unique())

Index(['match_id', 'round_id', 'name', 'slug', 'cup_round_type'], dtype='object')
['Round of 16' 'Quarterfinal' 'Round of 32' 'Semifinal'
 'Qualification round 1' 'Final' 'Round of 64' 'Qualification round 2'
 'Semifinals' 'Quarterfinals' 'Round of 128']


In [5]:
print(data["matchevent"].columns)

Index(['match_id', 'first_to_serve', 'home_team_seed', 'away_team_seed',
       'custom_id', 'winner_code', 'default_period_count', 'start_datetime',
       'match_slug', 'final_result_only'],
      dtype='object')


In [6]:
print(data["matchtournament"].columns)
print(data["matchhome_team"].columns)
print(data["matchaway_team"].columns)

Index(['match_id', 'tournament_id', 'tournament_name', 'tournament_slug',
       'tournament_unique_id', 'tournament_category_name',
       'tournament_category_slug', 'user_count', 'ground_type',
       'tennis_points', 'has_event_player_statistics',
       'crowd_sourcing_enabled', 'has_performance_graph_feature',
       'display_inverse_home_away_teams', 'priority', 'competition_type'],
      dtype='object')
Index(['match_id', 'name', 'slug', 'gender', 'user_count', 'residence',
       'birthplace', 'height', 'weight', 'plays', 'turned_pro',
       'current_prize', 'total_prize', 'player_id', 'current_rank',
       'name_code', 'country', 'full_name'],
      dtype='object')
Index(['match_id', 'name', 'slug', 'gender', 'user_count', 'residence',
       'birthplace', 'height', 'weight', 'plays', 'turned_pro',
       'current_prize', 'total_prize', 'player_id', 'current_rank',
       'name_code', 'country', 'full_name'],
      dtype='object')


In [7]:
print(data["matchevent"]["winner_code"].value_counts())

winner_code
1    16552
2    15452
Name: count, dtype: int64


In [9]:
print(data["matchevent"]["start_datetime"].head())
print(data["matchevent"]["start_datetime"].dtype)

0    1706878800
1    1706871600
2    1706810400
3    1706800800
4    1706794200
Name: start_datetime, dtype: int64
int64


In [ ]:
import pandas as pd
import numpy as np


event = data["matchevent"].copy()

home = data["matchhome_team"][["match_id", "full_name"]].rename(
    columns={"full_name": "home_player"}
)

away = data["matchaway_team"][["match_id", "full_name"]].rename(
    columns={"full_name": "away_player"}
)

tournament = data["matchtournament"][["match_id", "tournament_id"]]


df = (
    event.merge(home, on="match_id")
         .merge(away, on="match_id")
         .merge(tournament, on="match_id")
)


df["month"] = pd.to_datetime(
    df["start_datetime"],
    unit="s"
).dt.to_period("M")


df["winner"] = np.where(
    df["winner_code"] == 1,
    df["home_player"],
    df["away_player"]
)


df = df[df["winner_code"].isin([1, 2])]


wins = (
    df.groupby(["month", "winner"])["tournament_id"]
      .nunique()
      .reset_index(name="tournaments_won")
)


best = wins.loc[wins["tournaments_won"].idxmax()]

print("All results:")
print(wins)

print("\nPlayer with the most tournament wins in a single month:")
print(best)

All results:
        month                  winner  tournaments_won
0     2024-01    Abbagnato, Anastasia                1
1     2024-01           Albie, Audrey                1
2     2024-01  Alexandrova, Ekaterina                1
3     2024-01         Appleton, Emily                1
4     2024-01        Arutiunian, Erik                1
...       ...                     ...              ...
3314  2024-04       Wagner, Stephanie                1
3315  2024-04         Waldner, Niklas                1
3316  2024-04      Wong, Hong Yi Cody                1
3317  2024-04            Zhu, Michael                1
3318  2024-04       Šrámková, Rebecca                1

[3319 rows x 3 columns]

Player with the most tournament wins in a single month:
month                 2024-02
winner             Naw, Hazem
tournaments_won             7
Name: 1001, dtype: object
